# Phase 5 — Synthesis: the two-part answer

Assemble the master figures from the saved `results/*.jld2` and state the answer.

**The mixing time of the TNMH chain.**
1. **Worst case (exact).** `t_rel = C` exactly (`λ₂ = 1 − 1/C`, Phase 1), and
   `C ~ ρ(Ly,D,β)^{Lx}` with `log ρ` ∝ `Ly` at criticality (Phase 2) — i.e.
   **exponential in volume** for any fixed `D < 2^{Ly/2}` at `β_c`.
2. **Average case (explains practice).** `χ²(π‖q) = Var_q(w)` and the bulk of the
   weight distribution stay mild while `C` is set by an exponentially rare config;
   the restricted constant `C_S ≪ C` and the empirical `τ_int` track the typical
   case (Phase 3). Hence usable acceptance rates and fast equilibration in practice.

**Bond-dimension verdict.** Controlling the per-layer rate (`ρ ≤ 1 + 1/Lx`) needs
`D ~ poly(Ly)` (match `log D ≳ S_exact`); a strictly size-independent `C` (`ρ = 1`)
needs `D ~ 2^{Ly/2}` (Phase 2.5).


In [ ]:
using Plots, JLD2, Statistics, LinearAlgebra, DelimitedFiles, Printf
include("main.jl"); include("tools/tnmh_tools.jl")
mkpath("results")
set_seed(20240618)
betac = log(1 + sqrt(2)) / 2          # β_c = ln(1+√2)/2 ≈ 0.4407
log_finding(s) = open(io -> println(io, s), "results/FINDINGS.md", "a")
println("ready. β_c = ", round(betac, digits=6))


## Master figures F1–F5 (from saved data; run Phases 1–4 first)

In [ ]:
load_ok(f) = isfile(f) ? true : (println("missing $f — run its phase notebook"); false)

# F1: λ₂ vs 1−1/C
if load_ok("results/phase1_spectrum.jld2")
    d = jldopen("results/phase1_spectrum.jld2","r"); pred=d["pred"]; l2=d["lambda2"]; close(d)
    F1 = scatter(pred, l2, label="cases", xlabel="1 − 1/C", ylabel="λ₂",
                 title="F1: C is the relaxation time")
    lo, hi = minimum(pred), maximum(pred)
    plot!(F1, [lo,hi], [lo,hi], ls=:dash, label="y = x")
    savefig(F1, "results/F1_lambda2_vs_C.png"); display(F1)
end


In [ ]:
# F2: log C vs Lx (line) and log ρ vs Ly
if load_ok("results/phase2_scaling.jld2")
    d = jldopen("results/phase2_scaling.jld2","r")
    Lx_list=d["Lx_list"]; logC=d["logC"]; Ly_array=d["Ly_array"]; rho_D2=d["rho_D2"]; close(d)
    F2a = scatter(Lx_list, logC, label="log C", xlabel="L_x", ylabel="log C", title="F2a: C ~ ρ^{Lx}")
    F2b = plot(Ly_array, rho_D2, marker=:circle, yscale=:log10, label="ρ (D=2)",
               xlabel="L_y", ylabel="ρ", title="F2b: log ρ ∝ L_y")
    F2 = plot(F2a, F2b, layout=(1,2), size=(950,380)); savefig(F2, "results/F2_scaling.png"); display(F2)
end


In [ ]:
# F4: χ² vs C vs volume
if load_ok("results/phase3_typical.jld2")
    d = jldopen("results/phase3_typical.jld2","r"); vols=d["vols"]; chi2e=d["chi2_exact"]; Cs=d["C"]; close(d)
    F4 = plot(vols, max.(chi2e,1e-12), marker=:circle, lw=2, yscale=:log10, label="χ² (typical)",
              xlabel="volume N", ylabel="value", title="F4: typical χ² vs worst-case C")
    plot!(F4, vols, max.(Cs .- 1,1e-12), marker=:square, lw=2, label="C − 1 (worst case)")
    savefig(F4, "results/F4_chi2_vs_C.png"); display(F4)
end


## Write the synthesis into FINDINGS.md

In [ ]:
log_finding("\n## Phase 5 — Synthesis (master answer)")
log_finding("1. Worst case: t_rel = C exactly (Phase 1); C ~ ρ(Ly,D,β)^{Lx}, exponential in volume at β_c (Phase 2).")
log_finding("2. Average case: χ² and the weight bulk stay mild; C_S ≪ C; τ_int tracks the typical case (Phase 3) ⇒ fast in practice.")
log_finding("Bond-dimension verdict: ρ-control needs D~poly(Ly); size-independent C needs D~2^{Ly/2} (Phase 2.5).")
log_finding("Master figures: results/F1_lambda2_vs_C.png, F2_scaling.png, F4_chi2_vs_C.png.")
println("synthesis appended to results/FINDINGS.md")
